# Autoencoder — Anomaly Detection
Baseline · Tuned · Semi-supervised vs Unsupervised ablation

In [1]:
import random
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from itertools import product
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.append('..')
from src.autoencoder import DeepAutoencoder, _train_ae, _reconstruction_errors, _best_f1_threshold
from src.metrics import evaluate_predictions, save_metrics

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 1. Load preprocessed data

In [2]:
DATA_DIR = '../data/processed'

X_train_normal = pd.read_csv(f'{DATA_DIR}/X_train_normal.csv').values.astype(np.float32)
X_val_s        = pd.read_csv(f'{DATA_DIR}/X_val.csv').values.astype(np.float32)
X_test_s       = pd.read_csv(f'{DATA_DIR}/X_test.csv').values.astype(np.float32)
y_val          = pd.read_csv(f'{DATA_DIR}/y_val.csv').values.ravel()
y_test         = pd.read_csv(f'{DATA_DIR}/y_test.csv').values.ravel()

# 10% of normal training data held out for early stopping
rng           = np.random.default_rng(SEED)
idx           = rng.permutation(len(X_train_normal))
n_earlystop   = max(1, int(len(X_train_normal) * 0.1))
X_earlystop_s = X_train_normal[idx[:n_earlystop]]
X_train_s     = X_train_normal[idx[n_earlystop:]]

INPUT_DIM = X_train_s.shape[1]
print(f'Train (normal): {X_train_s.shape}')
print(f'Early-stop:     {X_earlystop_s.shape}')
print(f'Val:            {X_val_s.shape}  fraud={int(y_val.sum())}')
print(f'Test:           {X_test_s.shape}  fraud={int(y_test.sum())}')

Train (normal): (153530, 30)
Early-stop:     (17058, 30)
Val:            (56962, 30)  fraud=99
Test:           (56962, 30)  fraud=98


## 2. Baseline — normal-only training
Default architecture and hyperparameters.

In [3]:
baseline_config = {
    'hidden_dims':  (64, 32, 16, 8),
    'lr':           1e-3,
    'weight_decay': 1e-5,
    'batch_size':   256,
}

print('Training baseline (normal-only)...')
baseline_model, baseline_epochs, baseline_val_loss = _train_ae(
    baseline_config, X_train_s, X_earlystop_s, device
)

val_errors_bl  = _reconstruction_errors(baseline_model, X_val_s,  device)
test_errors_bl = _reconstruction_errors(baseline_model, X_test_s, device)

threshold_bl   = _best_f1_threshold(y_val, val_errors_bl)
metrics_bl     = evaluate_predictions(y_test, test_errors_bl, threshold=threshold_bl)

print(f'Epochs run: {baseline_epochs}')
print(f'Precision : {metrics_bl["precision"]:.4f}')
print(f'Recall    : {metrics_bl["recall"]:.4f}')
print(f'F1        : {metrics_bl["f1"]:.4f}')
print(f'ROC-AUC   : {metrics_bl["roc_auc"]:.4f}')
print(f'PR-AUC    : {metrics_bl["pr_auc"]:.4f}')

save_metrics(metrics_bl, model_name='autoencoder_baseline', path='../results/metrics.csv')

Training baseline (normal-only)...


Epochs run: 150
Precision : 0.6140
Recall    : 0.7143
F1        : 0.6604
ROC-AUC   : 0.9546
PR-AUC    : 0.6361


## 3. Grid search — normal-only training
12 configurations; select best by validation PR-AUC.

In [4]:
param_grid = {
    'hidden_dims':  [(64, 32, 16, 8), (128, 64, 32, 16), (32, 16, 8)],
    'weight_decay': [1e-5, 1e-4],
    'lr':           [1e-3],
    'batch_size':   [2048],
}

keys        = list(param_grid.keys())
all_configs = [dict(zip(keys, v)) for v in product(*param_grid.values())]
total       = len(all_configs)
print(f'Total configurations: {total}\n')

gs_results  = []
best_pr_auc = -1.0
best_config = None

for i, config in enumerate(all_configs, 1):
    model, epochs, _ = _train_ae(config, X_train_s, X_earlystop_s, device)
    val_errors       = _reconstruction_errors(model, X_val_s, device)
    pr_auc           = average_precision_score(y_val, val_errors)
    roc_auc          = evaluate_predictions(y_val, val_errors)['roc_auc']

    row = {
        'hidden_dims':  str(config['hidden_dims']),
        'weight_decay': config['weight_decay'],
        'lr':           config['lr'],
        'batch_size':   config['batch_size'],
        'epochs':       epochs,
        'val_pr_auc':   round(pr_auc, 4),
        'val_roc_auc':  round(roc_auc, 4),
    }
    gs_results.append(row)

    print(f'[{i:>2}/{total}] arch={str(config["hidden_dims"]):<22} '
          f'wd={config["weight_decay"]:.0e} bs={config["batch_size"]:<4} '
          f'-> val PR-AUC={pr_auc:.4f}  ROC-AUC={roc_auc:.4f}')

    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        best_config = config

gs_df = pd.DataFrame(gs_results).sort_values('val_pr_auc', ascending=False).reset_index(drop=True)
print('\nTop 5 configurations:')
print(gs_df.head(5).to_string(index=False))
print(f'\nBest config: {best_config}')

Total configurations: 6



[ 1/6] arch=(64, 32, 16, 8)        wd=1e-05 bs=2048 -> val PR-AUC=0.3497  ROC-AUC=0.9412


[ 2/6] arch=(64, 32, 16, 8)        wd=1e-04 bs=2048 -> val PR-AUC=0.4060  ROC-AUC=0.9418


[ 3/6] arch=(128, 64, 32, 16)      wd=1e-05 bs=2048 -> val PR-AUC=0.5893  ROC-AUC=0.9467


[ 4/6] arch=(128, 64, 32, 16)      wd=1e-04 bs=2048 -> val PR-AUC=0.5888  ROC-AUC=0.9452


[ 5/6] arch=(32, 16, 8)            wd=1e-05 bs=2048 -> val PR-AUC=0.5462  ROC-AUC=0.9455


[ 6/6] arch=(32, 16, 8)            wd=1e-04 bs=2048 -> val PR-AUC=0.5259  ROC-AUC=0.9476

Top 5 configurations:
      hidden_dims  weight_decay    lr  batch_size  epochs  val_pr_auc  val_roc_auc
(128, 64, 32, 16)       0.00001 0.001        2048     150      0.5893       0.9467
(128, 64, 32, 16)       0.00010 0.001        2048     150      0.5888       0.9452
      (32, 16, 8)       0.00001 0.001        2048     150      0.5462       0.9455
      (32, 16, 8)       0.00010 0.001        2048     150      0.5259       0.9476
  (64, 32, 16, 8)       0.00010 0.001        2048     150      0.4060       0.9418

Best config: {'hidden_dims': (128, 64, 32, 16), 'weight_decay': 1e-05, 'lr': 0.001, 'batch_size': 2048}


## 4. Tuned — normal-only training
Retrain the best config from grid search and evaluate on the test set.

In [5]:
print('Retraining best config (normal-only)...')
tuned_model, tuned_epochs, _ = _train_ae(
    best_config, X_train_s, X_earlystop_s, device
)

val_errors_tu  = _reconstruction_errors(tuned_model, X_val_s,  device)
test_errors_tu = _reconstruction_errors(tuned_model, X_test_s, device)

threshold_tu = _best_f1_threshold(y_val, val_errors_tu)
metrics_tu   = evaluate_predictions(y_test, test_errors_tu, threshold=threshold_tu)

print(f'Best config : {best_config}')
print(f'Epochs run  : {tuned_epochs}')
print(f'Precision   : {metrics_tu["precision"]:.4f}')
print(f'Recall      : {metrics_tu["recall"]:.4f}')
print(f'F1          : {metrics_tu["f1"]:.4f}')
print(f'ROC-AUC     : {metrics_tu["roc_auc"]:.4f}')
print(f'PR-AUC      : {metrics_tu["pr_auc"]:.4f}')

save_metrics(metrics_tu, model_name='autoencoder_tuned', path='../results/metrics.csv')

Retraining best config (normal-only)...


Best config : {'hidden_dims': (128, 64, 32, 16), 'weight_decay': 1e-05, 'lr': 0.001, 'batch_size': 2048}
Epochs run  : 150
Precision   : 0.6939
Recall      : 0.6939
F1          : 0.6939
ROC-AUC     : 0.9709
PR-AUC      : 0.6955


## 5. Ablation — baseline, unsupervised (all data)
Train on the full training set (normal + fraud) without any label filtering.
Preprocessing is done inline here since `preprocessing.py` only saves normal training data.

In [6]:
import kagglehub

path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')
df   = pd.read_csv(path + '/creditcard.csv').drop_duplicates()

X_raw = df.drop('Class', axis=1)
y_raw = df['Class']

X_tr_full, X_te, y_tr_full, y_te = train_test_split(
    X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=SEED
)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr_full, y_tr_full, test_size=0.25, stratify=y_tr_full, random_state=SEED
)

def log_amount(df):
    out = df.copy()
    out['Amount'] = np.log1p(out['Amount'])
    return out

X_tr_log = log_amount(X_tr)
X_va_log = log_amount(X_va)
X_te_log = log_amount(X_te)

# scaler fitted on all training data (unsupervised — no label filtering)
unsup_scaler = StandardScaler().fit(X_tr_log)

X_unsup_train_s = unsup_scaler.transform(X_tr_log).astype(np.float32)
X_unsup_val_s   = unsup_scaler.transform(X_va_log).astype(np.float32)
X_unsup_test_s  = unsup_scaler.transform(X_te_log).astype(np.float32)
y_unsup_val     = y_va.reset_index(drop=True).values
y_unsup_test    = y_te.reset_index(drop=True).values

# 10% early-stop split
rng_u           = np.random.default_rng(SEED)
idx_u           = rng_u.permutation(len(X_unsup_train_s))
n_es_u          = max(1, int(len(X_unsup_train_s) * 0.1))
X_unsup_es_s    = X_unsup_train_s[idx_u[:n_es_u]]
X_unsup_tr_s    = X_unsup_train_s[idx_u[n_es_u:]]

print(f'Unsup train (all data): {X_unsup_tr_s.shape}  '
      f'fraud in train={int(y_tr.sum())}')
print(f'Unsup early-stop:       {X_unsup_es_s.shape}')

C:\Users\qblcr\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsup train (all data): (153212, 30)  fraud in train=284
Unsup early-stop:       (17023, 30)


In [7]:
print('Training baseline (unsupervised — all data)...')
bl_unsup_model, bl_unsup_epochs, _ = _train_ae(
    baseline_config, X_unsup_tr_s, X_unsup_es_s, device
)

val_errors_bl_u  = _reconstruction_errors(bl_unsup_model, X_unsup_val_s,  device)
test_errors_bl_u = _reconstruction_errors(bl_unsup_model, X_unsup_test_s, device)

threshold_bl_u = _best_f1_threshold(y_unsup_val, val_errors_bl_u)
metrics_bl_u   = evaluate_predictions(y_unsup_test, test_errors_bl_u, threshold=threshold_bl_u)

print(f'Epochs run : {bl_unsup_epochs}')
print(f'Precision  : {metrics_bl_u["precision"]:.4f}')
print(f'Recall     : {metrics_bl_u["recall"]:.4f}')
print(f'F1         : {metrics_bl_u["f1"]:.4f}')
print(f'ROC-AUC    : {metrics_bl_u["roc_auc"]:.4f}')
print(f'PR-AUC     : {metrics_bl_u["pr_auc"]:.4f}')

save_metrics(metrics_bl_u, model_name='autoencoder_baseline_unsupervised', path='../results/metrics.csv')

Training baseline (unsupervised — all data)...


Epochs run : 150
Precision  : 0.1023
Recall     : 0.1895
F1         : 0.1328
ROC-AUC    : 0.9303
PR-AUC     : 0.0674


## 6. Tuned — unsupervised (all data)
Best config from the normal-only grid search, retrained on all data.

In [8]:
print('Retraining best config (unsupervised — all data)...')
tu_unsup_model, tu_unsup_epochs, _ = _train_ae(
    best_config, X_unsup_tr_s, X_unsup_es_s, device
)

val_errors_tu_u  = _reconstruction_errors(tu_unsup_model, X_unsup_val_s,  device)
test_errors_tu_u = _reconstruction_errors(tu_unsup_model, X_unsup_test_s, device)

threshold_tu_u = _best_f1_threshold(y_unsup_val, val_errors_tu_u)
metrics_tu_u   = evaluate_predictions(y_unsup_test, test_errors_tu_u, threshold=threshold_tu_u)

print(f'Best config: {best_config}')
print(f'Epochs run : {tu_unsup_epochs}')
print(f'Precision  : {metrics_tu_u["precision"]:.4f}')
print(f'Recall     : {metrics_tu_u["recall"]:.4f}')
print(f'F1         : {metrics_tu_u["f1"]:.4f}')
print(f'ROC-AUC    : {metrics_tu_u["roc_auc"]:.4f}')
print(f'PR-AUC     : {metrics_tu_u["pr_auc"]:.4f}')

save_metrics(metrics_tu_u, model_name='autoencoder_tuned_unsupervised', path='../results/metrics.csv')

Retraining best config (unsupervised — all data)...


Best config: {'hidden_dims': (128, 64, 32, 16), 'weight_decay': 1e-05, 'lr': 0.001, 'batch_size': 2048}
Epochs run : 150
Precision  : 0.1493
Recall     : 0.4211
F1         : 0.2204
ROC-AUC    : 0.9474
PR-AUC     : 0.1207


## 7. Summary — all variants

In [9]:
summary = pd.DataFrame([
    {'Model': 'AE Baseline (normal-only)',    **metrics_bl},
    {'Model': 'AE Tuned (normal-only)',       **metrics_tu},
    {'Model': 'AE Baseline (unsupervised)',   **metrics_bl_u},
    {'Model': 'AE Tuned (unsupervised)',      **metrics_tu_u},
])

summary = summary[['Model', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']]
summary = summary.round(4)
print(summary.to_string(index=False))

                     Model  precision  recall     f1  roc_auc  pr_auc
 AE Baseline (normal-only)     0.6140  0.7143 0.6604   0.9546  0.6361
    AE Tuned (normal-only)     0.6939  0.6939 0.6939   0.9709  0.6955
AE Baseline (unsupervised)     0.1023  0.1895 0.1328   0.9303  0.0674
   AE Tuned (unsupervised)     0.1493  0.4211 0.2204   0.9474  0.1207
